# 02 — QPIE Encoding + QHED (Yao et al. 2017)

Amplitude encoding, the canonical H–dec–H interference circuit, and the analytic edge probability. The circuit output is verified against the exact statevector.

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('')) if os.path.basename(os.getcwd())!='Quantum_Image' else '.')
sys.path.insert(0, '..') if os.path.basename(os.getcwd())!='Quantum_Image' else None
sys.path.insert(0, os.getcwd())
%matplotlib inline
import numpy as np
from data.synthetic.generator import BUILDERS
from quantum.qpie_encoder import build_qpie_circuit, image_to_amplitudes
from quantum.qhed_circuit import build_qhed_circuit, qhed_analytic_edge_probability
from qiskit.quantum_info import Statevector
img, _ = BUILDERS['circle'](8)
enc = img / img.max()
qc = build_qhed_circuit(enc, 'horizontal', measure=False)
sv = Statevector(qc)
# compare with the analytic P(a=1) map
probs = sv.probabilities_dict()
n = 8; k = 3
pa1 = np.zeros((n, n))
for key, p in probs.items():
    v = int(key, 2)
    if v & 1:
        pos = v >> 1
        pa1[pos % n, pos // n] += p
analytic = qhed_analytic_edge_probability(enc, 'horizontal')
print('max |circuit - analytic| =', np.abs(pa1 - analytic).max())
print('qubits:', qc.num_qubits, '| depth:', qc.depth())

In [ ]:

from quantum.quantum_pipeline import run_quantum
from quantum.threshold_selector import apply_threshold
res, rec = run_quantum(img, 'qhed', shots=8192)
print('resources:', {k: rec[k] for k in ('logical_qubits','raw_gates','raw_depth','kernel_time_s','e2e_time_s')})
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('input')
ax[1].imshow(res.strength, cmap='magma'); ax[1].set_title('QHED strength (8192 shots)')
ax[2].imshow(apply_threshold(res.strength, 0.2), cmap='gray'); ax[2].set_title('binary (rel t=0.2)')
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.savefig('../results/figures/nb02_qhed.png', dpi=130)